In [ ]:
# @title Imports

import os

from matplotlib import pyplot as plt
import numpy as np

from perch_hoplite.agile import audio_loader
from perch_hoplite.agile import classifier
from perch_hoplite.agile import classifier_data
from perch_hoplite.agile import embedding_display
from perch_hoplite.agile import source_info
from perch_hoplite.db  import brutalism
from perch_hoplite.db import score_functions
from perch_hoplite.db  import search_results
from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.zoo import model_configs
from perch_hoplite.zoo import taxonomy_model_tf

In [ ]:
# @title Load model and connect to database {vertical-output: true}

# @markdown Location of database containing audio embeddings.
db_path = '/home/reindert/Valentin_REVO/perch_hoplite/Data/db/'  # @param {type:'string'}

# @markdown Identifier (e.g. name) to attach to labels produced during validation.
annotator_id = 'valentin'  # @param {type: 'string'}

db = sqlite_usearch_impl.SQLiteUSearchDB.create(db_path)
db_model_config = db.get_metadata('model_config')
embed_config = db.get_metadata('audio_sources')
model_class = model_configs.get_model_class(db_model_config.model_key)
embedding_model = model_class.from_config(db_model_config.model_config)
audio_sources = source_info.AudioSources.from_config_dict(embed_config)
if hasattr(embedding_model, 'window_size_s'):
  window_size_s = embedding_model.window_size_s
else:
  window_size_s = 5.0
audio_filepath_loader = audio_loader.make_filepath_loader(
    audio_sources=audio_sources,
    window_size_s=window_size_s,
    sample_rate_hz=embedding_model.sample_rate,
)

# Search

In [ ]:
# Reference sounds list
ref_sound_path = '/home/reindert/Valentin_REVO/perch_hoplite/Data/ref_sound/'
ref_sound_list = os.listdir(ref_sound_path)

print(ref_sound_list)

In [ ]:
# @title Load query audio {vertical-output: true}

# @markdown The `query_uri` can be a URL, filepath, or Xeno-Canto ID
# @markdown (like `xc105133`, containing a Wood Thrush (`woothr`)).
query_uri = '/home/reindert/Valentin_REVO/perch_hoplite/Data/ref_sound/squak/squak.wav'  # @param {type: 'string'}
query_label = 'squak'  # @param {type: 'string'}

query = embedding_display.QueryDisplay(
    uri=query_uri, offset_s=0.0, window_size_s=5.0, sample_rate_hz=32000)
_ = query.display_interactive()

In [ ]:
# @title Embed the Query and Search {vertical-output: true}

# @markdown Number of results to find and display.
num_results = 25  # @param
query_embedding = embedding_model.embed(
    query.get_audio_window()).embeddings[0, 0]

# @markdown If checked, search for examples near a particular target score.
target_sampling = False  # @param {type: 'boolean'}

# @markdown When target sampling, target this score.
target_score = -1.0  # @param
if not target_sampling:
  target_score = None

# @markdown If True, search the full DB. Otherwise, use approximate
# @markdown nearest-neighbor search.
exact_search = False  # @param {type: 'boolean'}

results = db.search(
    query_embedding,
    search_list_size=num_results,
    approximate=exact_search,
    target_score=target_score,
)
# Get a random batch of scores to plot the score distribution.
scores = brutalism.get_random_embedding_scores(
    db, query_embedding, score_fn=score_functions.get_score_fn('dot'),
    sample_size=2_048,
    rng_seed=42,
)
_ = plt.hist(scores, bins=25, density=True, alpha=0.5)
hit_scores = [r.sort_score for r in results.search_results]
plt.scatter(hit_scores, np.zeros_like(hit_scores), marker='|',
            color='r', alpha=0.5)


In [ ]:
# @title Display Results {vertical-output: true}

display_results = embedding_display.EmbeddingDisplayGroup.from_search_results(
    results,
    db,
    sample_rate_hz=32000,
    frame_rate=100,
    audio_loader=audio_filepath_loader,
)
display_results.display(positive_labels=[query_label], paged_mode=False)

In [ ]:
# @title Save data labels {vertical-output: true}

print("Annotations before saving new labels:", len(db.get_all_annotations()))

db.insert_annotations(
    display_results.harvest_labels(annotator_id),
    handle_duplicates="skip",
)

print("Annotations after saving new labels:", len(db.get_all_annotations()))

# Classify

In [ ]:
# @title Classifier training {vertical-output: true}

# @markdown Set of labels to classify. If None, auto-populated from the DB.
target_labels = None  # @param

# @markdown Classifier traning hyperparams. These should not require tuning.
learning_rate = 1e-3  # @param
weak_neg_weight = 0.05  # @param
l2_mu = 0.000  # @param
num_steps = 128  # @param

train_ratio = 0.9  # @param
batch_size = 128  # @param
weak_negatives_batch_size = 128  # @param
loss_fn_name = 'bce'  # @param ['hinge', 'bce']

data_manager = classifier_data.AgileDataManager(
    target_labels=target_labels,
    db=db,
    train_ratio=train_ratio,
    min_eval_examples=1,
    batch_size=batch_size,
    weak_negatives_batch_size=weak_negatives_batch_size,
    rng=np.random.default_rng(seed=5))
print('Training for target labels : ')
print(data_manager.get_target_labels())
linear_classifier, eval_scores = classifier.train_linear_classifier(
    data_manager=data_manager,
    learning_rate=learning_rate,
    weak_neg_weight=weak_neg_weight,
    num_train_steps=num_steps,
)
print('\n' + '-' * 80)
top1 = eval_scores['top1_acc']
print(f'top-1      {top1:.3f}')
rocauc = eval_scores['roc_auc']
print(f'roc_auc    {rocauc:.3f}')
cmap = eval_scores['cmap']
print(f'cmap       {cmap:.3f}')

# Save linear classifier.
linear_classifier.save(os.path.join(db_path, 'agile_classifier_v2.pt'))

In [ ]:
# @title Review Classifier Results {vertical-output: true}

# @markdown Number of results to find and display.
target_label = 'dj'  # @param {type: 'string'}
num_results = 50  # @param

target_label_idx = data_manager.get_target_labels().index(target_label)
class_query = linear_classifier.beta[:, target_label_idx]
bias = linear_classifier.beta_bias[target_label_idx]

# @markdown Number of (randomly selected) database entries to search over.
sample_size = 1_000_000  # @param

# @markdown Whether to use margin-sampling. If checked, search for examples
# @markdown with logits near a particular target score (usually 0).
margin_sampling = True  # @param {type: 'boolean'}

# @markdown When margin sampling, target this logit.
margin_target_score = -1.0  # @param
if not margin_sampling:
  margin_target_score = None
score_fn = score_functions.get_score_fn(
    'dot', bias=bias, target_score=margin_target_score)
results = brutalism.threaded_brute_search(
    db, class_query, num_results, score_fn=score_fn,
    sample_size=sample_size)

# Get a random batch of scores to plot the score distribution.
scores = brutalism.get_random_embedding_scores(
    db,
    class_query,
    score_fn=score_functions.get_score_fn('dot', bias=bias),
    sample_size=2_048,
    rng_seed=42,
)
plt.hist(scores, bins=25, density=True, alpha=0.5)
hit_scores = [r.sort_score for r in results.search_results]
_ = plt.scatter(hit_scores, np.zeros_like(hit_scores), marker='|',
            color='r', alpha=0.5)


In [ ]:
# @title Display Results {vertical-output: true}

display_results = embedding_display.EmbeddingDisplayGroup.from_search_results(
    results,
    db,
    sample_rate_hz=32000,
    frame_rate=100,
    audio_loader=audio_filepath_loader,
)
display_results.display(positive_labels=[target_label, 'dj'], paged_mode=False)

In [ ]:
# @title Save data labels {vertical-output: true}

print("Annotations before saving new labels:", len(db.get_all_annotations()))

db.insert_annotations(
    display_results.harvest_labels(annotator_id),
    handle_duplicates="overwrite",
)

print("Annotations after saving new labels:", len(db.get_all_annotations()))

In [ ]:
# @title Run inference with trained classifier {vertical-output: true}

output_csv_filepath = '/home/reindert/Valentin_REVO/perch_hoplite/Data/inference_saskia_100626.csv'  # @param {type: 'string'}
logit_threshold = 1.0  # @param
# Set labels to a tuple of desired labels if you want to run inference on
# a subset of the labels.
labels = None  # @param

classifier.write_inference_csv(
    linear_classifier,
    db,
    output_csv_filepath,
    logit_threshold,
    labels=labels,
    window_ids=db.match_window_ids(),
)

## Plotting

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Your existing df
df = pd.read_csv(output_csv_filepath)

# Get all wav files
wav_files = sorted([file for file in os.listdir('/home/reindert/Valentin_REVO/perch_hoplite/Data/HT2026/Location1') if file.endswith('.wav')])

# Count detections per filename (just the basename)
df['basename'] = df['filename'].apply(lambda x: os.path.basename(x))
counts = df.groupby('basename').size().reset_index(name='count')

# Build full list with zeros for missing files
all_files_df = pd.DataFrame({'basename': wav_files})
result = all_files_df.merge(counts, on='basename', how='left').fillna(0)
result['count'] = result['count'].astype(int)

# Parse datetime from filename pattern HHmmss_YYYYMMDD_*.wav
def parse_datetime(filename):
    parts = filename.split('_')
    time_str = parts[0]  # HHmmss
    date_str = parts[1]  # YYYYMMDD
    return datetime.strptime(date_str + time_str, '%Y%m%d%H%M%S')

result['datetime'] = result['basename'].apply(parse_datetime)
result = result.sort_values('datetime')

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(result['datetime'], result['count'], width=0.03, color='steelblue', edgecolor='none')

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
ax.xaxis.set_minor_formatter(mdates.DateFormatter('%H:%M'))

ax.tick_params(axis='x', which='major', labelsize=10, pad=15, rotation=0)
ax.tick_params(axis='x', which='minor', labelsize=7)
ax.set_ylabel('Number of detections')
ax.set_title('Detections per recording (HT2026 - Location1)')
plt.tight_layout()
plt.show()

print(result[['basename', 'datetime', 'count']])

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np

# Get all wav files
wav_files = sorted([file for file in os.listdir('/home/reindert/Valentin_REVO/perch_hoplite/Data/HT2026/Location1') if file.endswith('.wav')])

# Count detections per filename and label
df['basename'] = df['filename'].apply(lambda x: os.path.basename(x))
counts = df.groupby(['basename', 'label']).size().reset_index(name='count')

# Build full combination of all files x all labels with zeros for missing
labels = sorted(df['label'].unique())
all_combinations = pd.MultiIndex.from_product([wav_files, labels], names=['basename', 'label'])
result = all_combinations.to_frame(index=False)
result = result.merge(counts, on=['basename', 'label'], how='left').fillna(0)
result['count'] = result['count'].astype(int)

# Parse datetime
def parse_datetime(filename):
    parts = filename.split('_')
    return datetime.strptime(parts[1] + parts[0], '%Y%m%d%H%M%S')

result['datetime'] = result['basename'].apply(parse_datetime)
result = result.sort_values(['datetime', 'label'])

# Pivot so each label is a column
pivot = result.pivot_table(index='datetime', columns='label', values='count', fill_value=0)

# Plot stacked bars using integer index to avoid datetime width issues
fig, ax = plt.subplots(figsize=(14, 5))

bar_width = 0.8
colors = plt.cm.tab10.colors
x = np.arange(len(pivot))

bottom = np.zeros(len(pivot))
for i, label in enumerate(pivot.columns):
    ax.bar(x, pivot[label], width=bar_width, bottom=bottom,
           label=label, color=colors[i % len(colors)], edgecolor='none')
    bottom += pivot[label].values

# Label every Nth tick to avoid crowding
N = max(1, len(pivot) // 20)
tick_positions = x[::N]
tick_labels = [dt.strftime('%b %d\n%H:%M') for dt in pivot.index[::N]]

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, fontsize=8)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_ylabel('Number of detections')
ax.set_title('Detections per recording by label (HT2026 - Location1)')
ax.legend(title='Label')

plt.tight_layout()
plt.show()